# Finland (FI-TW) — Preprocessing notebook

**Period:** 2024-01-01 → 2024-06-30

**Raw inputs** (`Data/Finland/Raw/`):
- `matched_data_2024_*.csv` — six monthly FI-TW files; one row per train, with all stops packed into a serialized `timeTableRows` column. Each stop dict includes a nested `weather_observations` dict from FMI.

**Outputs** (`Data/Finland/processed/`): the five standardized CSVs of `utils.SCHEMA`.

Heavy logic — `ast.literal_eval` of `timeTableRows`, weather flattening, FI-TW imputation conventions — lives in `preprocess/lib_finland.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "utils.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "preprocess"))

import lib_finland as lib
from utils import SCHEMA, save_csv

RAW = ROOT / "Data" / "Finland" / "Raw"
OUT = ROOT / "Data" / "Finland" / "processed"
FIG = ROOT / "preprocess" / "figures" / "finland"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(42)

def save_fig(fig, name: str):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

## Step 1 — Glob and raw load

Six monthly `matched_data_2024_*.csv` files. Each row is one *train*, not one stop. We summarise per-file row counts and missing-value rates before the explosion.

In [ ]:
import glob
files = sorted(glob.glob(str(RAW / "matched_data_2024_*.csv")))
preview_rows = []
for path in files:
    raw = pd.read_csv(path, dtype=str, low_memory=False, nrows=200_000)
    preview_rows.append({"file": Path(path).name,
                          "rows_sampled": len(raw),
                          "nan_rate": float(raw.isna().mean().mean())})
preview_df = pd.DataFrame(preview_rows)
preview_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(preview_df["file"].str[-10:-4], preview_df["rows_sampled"], color="#0277BD")
axes[0].set_title("Rows per monthly file (sampled)"); axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(preview_df["file"].str[-10:-4], preview_df["nan_rate"], color="#90A4AE")
axes[1].set_title("Mean NaN rate (sampled)"); axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Step 1 — Raw file inventory", fontweight="bold")
save_fig(fig, "step_01_raw_inventory"); plt.show()

## Step 2 — Parse `timeTableRows` and explode

Each train carries a serialized list of stops in `timeTableRows`. We parse with `ast.literal_eval` (after a regex `nan→None` substitution to handle NumPy's repr leak) and explode to one row per stop. The pie chart shows parse outcomes per file; the histogram shows stops-per-train.

In [ ]:
df, parse_log = lib.load_and_explode(RAW)
print(f"After explosion: {len(df):,} stop rows")
parse_log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
totals = parse_log[["n_success", "n_empty", "n_error"]].sum()
axes[0].pie(totals, labels=totals.index, autopct="%1.1f%%",
            colors=["#43A047", "#FFA000", "#E53935"])
axes[0].set_title("timeTableRows parse outcomes")

if "train_number" in df.columns and "departure_date" in df.columns:
    stops_per = df.groupby(["train_number", "departure_date"]).size()
else:
    stops_per = pd.Series(parse_log["n_stops"] / parse_log["n_trains"].clip(lower=1))
axes[1].hist(stops_per.clip(upper=60), bins=40, color="#0277BD", edgecolor="white")
axes[1].set_title("Stops per train"); axes[1].set_xlabel("# stops (clipped at 60)")
fig.suptitle("Step 2 — Parse + explode", fontweight="bold")
save_fig(fig, "step_02_parse_explode"); plt.show()

## Step 3 — Type coercion

Numeric (delay, 9 weather variables) and boolean (`cancelled`) coercion. The bar chart shows the NaN rate per column **after** numeric conversion — values that failed to coerce.

In [ ]:
df, nan_log = lib.coerce_types(df)
nan_log

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(nan_log))
ax.bar(x - 0.2, nan_log["nan_rate_before"], 0.4, color="#90A4AE", label="before")
ax.bar(x + 0.2, nan_log["nan_rate"],        0.4, color="#0277BD", label="after coerce")
ax.set_xticks(x); ax.set_xticklabels(nan_log["column"], rotation=30)
ax.set_ylabel("NaN rate"); ax.set_title("Step 3 — Type coercion: NaN rate per column", fontweight="bold")
ax.legend()
save_fig(fig, "step_03_coercion"); plt.show()

## Step 4 — Date filter

Strict `[2024-01-01, 2024-06-30]` window.

In [ ]:
before = len(df)
df, info = lib.filter_dates(df)
print(info)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["departure_date"].dropna(), bins=24, color="#0277BD", edgecolor="white")
ax.set_title(f"Step 4 — departure_date distribution after filter ({before:,}→{info['rows_after']:,})",
             fontweight="bold")
ax.set_xlabel("date"); ax.set_ylabel("row count")
save_fig(fig, "step_04_date_filter"); plt.show()

## Step 5 — Cancellation flag breakdown

After the explosion, the stop-level `cancelled` field can come from the train OR the stop. We show the resulting True/False/NaN distribution.

In [ ]:
cancel_counts = df["cancelled"].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(cancel_counts, labels=[str(x) for x in cancel_counts.index],
        autopct="%1.2f%%", colors=["#43A047", "#E53935", "#FFA000"])
ax.set_title("Step 5 — cancelled flag distribution", fontweight="bold")
save_fig(fig, "step_05_cancellation"); plt.show()

## Step 6 — Weather imputation

FI-TW convention: `precipitation_1h` and `snow_depth` → 0 when missing (no measurement = no event); `air_temp` and `visibility` → monthly median, with the global median as final fallback. Before/after histograms confirm the strategy doesn't distort the distribution.

In [ ]:
before_air = df["air_temp"].copy() if "air_temp" in df.columns else None
before_vis = df["visibility"].copy() if "visibility" in df.columns else None

df, imp_log = lib.impute_weather(df)
imp_log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if before_air is not None:
    axes[0].hist(before_air.dropna(), bins=60, alpha=0.6, color="#90A4AE", label="before")
    axes[0].hist(df["air_temp"].dropna(), bins=60, alpha=0.6, color="#0277BD", label="after")
    axes[0].set_title("air_temp before/after imputation"); axes[0].legend()
if before_vis is not None:
    axes[1].hist(before_vis.dropna(), bins=60, alpha=0.6, color="#90A4AE", label="before")
    axes[1].hist(df["visibility"].dropna(), bins=60, alpha=0.6, color="#0277BD", label="after")
    axes[1].set_title("visibility before/after imputation"); axes[1].legend()
fig.suptitle("Step 6 — Weather imputation", fontweight="bold")
save_fig(fig, "step_06_imputation"); plt.show()

## Step 7 — Snow depth distribution

Boxplot per month + a station map coloured by max snow depth — should highlight the north.

In [ ]:
df = lib.normalize_codes(df)
df["month"] = df["departure_date"].dt.month

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df, x="month", y="snow_depth", ax=axes[0],
            color="#0277BD", showfliers=False)
axes[0].set_title("snow_depth by month")

# Station-map subplot is gated on lat/lon presence: per-stop rows in this
# pipeline don't carry coords (they're attached only later via nodes_station).
if {"lat", "lon"}.issubset(df.columns):
    snow_by_sta = (df.groupby("station_code")
                     .agg(snow_max=("snow_depth", "max"),
                          lat=("lat", "first"),
                          lon=("lon", "first")).dropna())
    sc = axes[1].scatter(snow_by_sta["lon"], snow_by_sta["lat"],
                          c=snow_by_sta["snow_max"], cmap="Blues", s=22,
                          edgecolor="k", linewidth=0.2)
    plt.colorbar(sc, ax=axes[1], label="max snow depth (cm)")
    axes[1].set_title("FI station map — max snow depth")
else:
    axes[1].text(0.5, 0.5, "lat/lon attached at build_nodes_station stage",
                  ha="center", va="center", transform=axes[1].transAxes,
                  fontsize=11, color="#777")
    axes[1].set_axis_off()
fig.suptitle("Step 7 — Snow distribution", fontweight="bold")
save_fig(fig, "step_07_snow_distribution"); plt.show()

## Step 8 — Weather severity ordinal

Compute the 0–4 ordinal from precipitation, snow depth and visibility; cross-tab against disruption rate to confirm the gradient is monotonic.

In [ ]:
df["weather_severity"] = lib.compute_weather_severity(df)
sev_counts = df["weather_severity"].value_counts().sort_index()

df["_late"] = (df["delay_minutes"] > lib.DISRUPTION_THRESHOLD) | df["cancelled"]
rate_by_sev = df.groupby("weather_severity")["_late"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(sev_counts.index.astype(str), sev_counts.values,
            color=sns.color_palette("Blues", 5))
axes[0].set_title("Severity ordinal counts"); axes[0].set_xlabel("severity 0–4")
axes[1].bar(rate_by_sev.index.astype(str), rate_by_sev.values,
            color=sns.color_palette("Reds", 5))
axes[1].set_title("P(disrupted | severity)"); axes[1].set_ylim(0, 1)
fig.suptitle("Step 8 — Weather severity", fontweight="bold")
save_fig(fig, "step_08_severity"); plt.show()

## Step 9 — Build `nodes_station` and FI station map

In [ ]:
nodes_station = lib.build_nodes_station(df)
print(f"{len(nodes_station):,} stations | missing coords: {nodes_station['lat'].isna().sum()}")

geo = nodes_station.dropna(subset=["lat", "lon"])
fig, ax = plt.subplots(figsize=(7, 9))
sc = ax.scatter(geo["lon"], geo["lat"],
                c=geo["avg_historical_delay"].clip(0, 10),
                s=10 + geo["degree"].clip(upper=40), cmap="Reds",
                edgecolor="k", linewidth=0.2)
plt.colorbar(sc, ax=ax, label="avg delay (min)")
ax.set_title("Step 9 — Finland station map\n(size = degree, colour = avg delay)", fontweight="bold")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
save_fig(fig, "step_09_station_map"); plt.show()

## Step 10 — Service labelling

In [ ]:
nodes_service = lib.build_nodes_service(df)
print(f"services: {len(nodes_service):,} | disruption rate = {nodes_service['is_disrupted'].mean()*100:.2f}%")

by_class = nodes_service.groupby("train_class_code")["is_disrupted"].mean().reset_index()
by_dow = (
    nodes_service.assign(dow=pd.to_datetime(nodes_service["date"]).dt.day_name())
                  .groupby("dow")["is_disrupted"].mean()
                  .reindex(["Monday","Tuesday","Wednesday","Thursday",
                            "Friday","Saturday","Sunday"])
)
by_month = (
    nodes_service.assign(m=pd.to_datetime(nodes_service["date"]).dt.month)
                  .groupby("m")["is_disrupted"].mean()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].bar(by_class["train_class_code"].astype(str), by_class["is_disrupted"], color="#0277BD")
axes[0].set_title("by train class"); axes[0].set_ylim(0, 1)
axes[1].bar(by_dow.index, by_dow.values, color="#0277BD")
axes[1].set_title("by day of week"); axes[1].set_ylim(0, 1); axes[1].tick_params(axis="x", rotation=20)
axes[2].plot(by_month.index, by_month.values, marker="o", color="#E91E63")
axes[2].set_title("by month"); axes[2].set_xticks(range(1,7)); axes[2].set_ylim(0, 1)
fig.suptitle("Step 10 — Service-level disruption rates", fontweight="bold")
save_fig(fig, "step_10_service_rates"); plt.show()

## Step 11 — Adjacency inference

FI has no mileage file, so `distance_km = 0` and adjacency comes from sorting consecutive stops by `scheduled_time`.

In [ ]:
sid_map = {c: f"FI_{c}" for c in df["station_code"].dropna().unique()}
edges_adjacent = lib.build_edges_adjacent(df, sid_map)
print(f"edges: {len(edges_adjacent):,} | distance_km == 0: {(edges_adjacent['distance_km']==0).all()}")

edges_per_station = edges_adjacent.groupby("station_from").size()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(edges_per_station.clip(upper=20), bins=20, color="#0277BD", edgecolor="white")
ax.set_title("Step 11 — Out-degree per station (inferred adjacency)", fontweight="bold")
ax.set_xlabel("# unique outgoing neighbours")
save_fig(fig, "step_11_adjacency"); plt.show()

## Step 12 — Persist all five standardized CSVs

In [ ]:
edges_stops_at = lib.build_edges_stops_at(df, sid_map)
nodes_fault    = lib.build_nodes_fault_empty()

save_csv(nodes_station,  OUT / "nodes_station.csv",  "nodes_station")
save_csv(nodes_service,  OUT / "nodes_service.csv",  "nodes_service")
save_csv(edges_stops_at, OUT / "edges_stops_at.csv", "edges_stops_at")
save_csv(edges_adjacent, OUT / "edges_adjacent.csv", "edges_adjacent")
save_csv(nodes_fault,    OUT / "nodes_fault.csv",    "nodes_fault")

summary = pd.DataFrame([
    {"file": "nodes_station.csv",  "rows": len(nodes_station)},
    {"file": "nodes_service.csv",  "rows": len(nodes_service)},
    {"file": "edges_stops_at.csv", "rows": len(edges_stops_at)},
    {"file": "edges_adjacent.csv", "rows": len(edges_adjacent)},
    {"file": "nodes_fault.csv",    "rows": len(nodes_fault)},
])
summary

In [ ]:
for name, dfx in [("nodes_station", nodes_station), ("nodes_service", nodes_service),
                   ("edges_stops_at", edges_stops_at), ("edges_adjacent", edges_adjacent),
                   ("nodes_fault", nodes_fault)]:
    expected = set(SCHEMA[name])
    actual   = set(dfx.columns)
    assert expected.issubset(actual) or len(dfx) == 0, f"{name}: missing {expected - actual}"
print("✓ Schema validation passed for all five outputs.")

## Closing summary

Finland preprocessing complete. Outputs are in `Data/Finland/processed/`; inspection plots in `preprocess/figures/finland/`. The unified stop-level pipeline consumes these next.